## Start GROBID Server Before Running the Benchmark

GROBID must be running before the benchmark cell is executed.

Open a **Terminal** in JupyterHub and copy/paste the following commands:

```bash
cd /home/jovyan/grobid-0.9.0
export JAVA_HOME=/opt/conda
export PATH=$JAVA_HOME/bin:$PATH
./gradlew :grobid-service:run
```

Leave this terminal running while the benchmark notebook runs.

Then return to this notebook and run the next cell to confirm GROBID is alive.

In [1]:
import requests

r = requests.get("http://localhost:8070/api/isalive", timeout=10)
print(r.status_code)
print(r.text)

200
true


In [3]:
from pathlib import Path
import sys
import pandas as pd

# Make sure Python can import /home/jovyan/parsers
ROOT = Path("/home/jovyan")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from parsers import run_parsers, PARSER_SPECS

INPUTS = ROOT / "inputs"
OUTPUTS = ROOT / "outputs"

PDF_PATHS = [
    INPUTS / "slam_llm.pdf",
    INPUTS / "cse188_research_paper.pdf",
    INPUTS / "cse195_research_paper.pdf",
]

SELECTED_PARSERS = [
    "pymupdf",
    "pdfplumber",
    "pypdf",
    "tika",
    "unstructured",
    "docling",
    "marker",
    "mineru",
    "grobid",
]

print("ROOT:", ROOT)
print("Current working directory:", Path.cwd())
print("Registered parsers:", list(PARSER_SPECS.keys()))
print("Selected parsers:", SELECTED_PARSERS)

missing_from_registry = [
    parser_name
    for parser_name in SELECTED_PARSERS
    if parser_name not in PARSER_SPECS
]

if missing_from_registry:
    raise ValueError(f"These selected parsers are not registered: {missing_from_registry}")

all_status_rows = []

for PDF_PATH in PDF_PATHS:
    if not PDF_PATH.exists():
        raise FileNotFoundError(f"Input PDF not found: {PDF_PATH}")

    RUN_OUTPUTS = OUTPUTS / PDF_PATH.stem
    PARSER_OUTPUTS = RUN_OUTPUTS / "parser_outputs"

    print("\n" + "=" * 100)
    print("Running benchmark for:", PDF_PATH)
    print("Output directory:", PARSER_OUTPUTS)
    print("=" * 100)

    parser_outputs, parser_status = run_parsers(
        PDF_PATH,
        output_dir=PARSER_OUTPUTS,
        selected=SELECTED_PARSERS,
    )

    df_parser_status = pd.DataFrame(parser_status)
    df_parser_status.insert(0, "document", PDF_PATH.name)

    status_path = RUN_OUTPUTS / "parser_status.csv"
    RUN_OUTPUTS.mkdir(parents=True, exist_ok=True)
    df_parser_status.to_csv(status_path, index=False)

    print("Saved parser status to:", status_path)
    print("Successful parsers:", list(parser_outputs.keys()))

    all_status_rows.append(df_parser_status)

df_all_status = pd.concat(all_status_rows, ignore_index=True)

combined_status_path = OUTPUTS / "combined_parser_status.csv"
df_all_status.to_csv(combined_status_path, index=False)

print("\nCombined parser status saved to:", combined_status_path)

df_all_status

ROOT: /home/jovyan
Current working directory: /home/jovyan/tests
Registered parsers: ['pymupdf', 'pdfplumber', 'pypdf', 'tika', 'unstructured', 'mineru', 'docling', 'marker', 'grobid']
Selected parsers: ['pymupdf', 'pdfplumber', 'pypdf', 'tika', 'unstructured', 'docling', 'marker', 'mineru', 'grobid']

Running benchmark for: /home/jovyan/inputs/slam_llm.pdf
Output directory: /home/jovyan/outputs/slam_llm/parser_outputs


/opt/conda/lib/python3.12/site-packages/tika/__init__.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)
No languages specified, defaulting to English.
[INFO] 2026-05-05 23:23:37,941 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-05 23:23:37,945 [RapidOCR] download_file.py:60: File exists and is valid: /opt/conda/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-05 23:23:37,945 [RapidOCR] main.py:57: Using /opt/conda/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-05 23:23:37,988 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-05 23:23:37,990 [RapidOCR] download_file.py:60: File exists and is valid: /opt/conda/lib/p

Saved parser status to: /home/jovyan/outputs/slam_llm/parser_status.csv
Successful parsers: ['pymupdf', 'pdfplumber', 'pypdf', 'tika', 'unstructured', 'docling', 'marker', 'mineru', 'grobid']

Running benchmark for: /home/jovyan/inputs/cse188_research_paper.pdf
Output directory: /home/jovyan/outputs/cse188_research_paper/parser_outputs


No languages specified, defaulting to English.
[INFO] 2026-05-05 23:25:00,795 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-05 23:25:00,797 [RapidOCR] download_file.py:60: File exists and is valid: /opt/conda/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-05 23:25:00,797 [RapidOCR] main.py:57: Using /opt/conda/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-05 23:25:00,839 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-05 23:25:00,840 [RapidOCR] download_file.py:60: File exists and is valid: /opt/conda/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-05 23:25:00,841 [RapidOCR] main.py:57: Using /opt/conda/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-05 23:25:00,857 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-05 23:25:00,862 [RapidOCR] down

Saved parser status to: /home/jovyan/outputs/cse188_research_paper/parser_status.csv
Successful parsers: ['pymupdf', 'pdfplumber', 'pypdf', 'tika', 'unstructured', 'docling', 'marker', 'mineru', 'grobid']

Running benchmark for: /home/jovyan/inputs/cse195_research_paper.pdf
Output directory: /home/jovyan/outputs/cse195_research_paper/parser_outputs


No languages specified, defaulting to English.
[INFO] 2026-05-05 23:25:56,746 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-05 23:25:56,748 [RapidOCR] download_file.py:60: File exists and is valid: /opt/conda/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-05 23:25:56,749 [RapidOCR] main.py:57: Using /opt/conda/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-05 23:25:56,789 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-05 23:25:56,789 [RapidOCR] download_file.py:60: File exists and is valid: /opt/conda/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-05 23:25:56,790 [RapidOCR] main.py:57: Using /opt/conda/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-05 23:25:56,807 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-05 23:25:56,811 [RapidOCR] down

Saved parser status to: /home/jovyan/outputs/cse195_research_paper/parser_status.csv
Successful parsers: ['pymupdf', 'pdfplumber', 'pypdf', 'tika', 'unstructured', 'docling', 'marker', 'mineru', 'grobid']

Combined parser status saved to: /home/jovyan/outputs/combined_parser_status.csv


,document,parser,status,chars,runtime_seconds,type,requires,error,output_path
0,slam_llm.pdf,pymupdf,success,88199,0.08,flat_text,,,/home/jovyan/outputs/slam_llm/parser_outputs/p...
1,slam_llm.pdf,pdfplumber,success,85465,1.80,flat_text,,,/home/jovyan/outputs/slam_llm/parser_outputs/p...
2,slam_llm.pdf,pypdf,success,88394,0.53,flat_text,,,/home/jovyan/outputs/slam_llm/parser_outputs/p...
3,slam_llm.pdf,tika,success,90660,0.11,semi_structured,"tika, java",,/home/jovyan/outputs/slam_llm/parser_outputs/t...
4,slam_llm.pdf,unstructured,success,95836,5.07,structure_aware,unstructured,,/home/jovyan/outputs/slam_llm/parser_outputs/u...
5,slam_llm.pdf,docling,success,100840,10.59,structure_aware,docling,,/home/jovyan/outputs/slam_llm/parser_outputs/d...
6,slam_llm.pdf,marker,success,102133,40.08,structure_aware,marker-pdf,,/home/jovyan/outputs/slam_llm/parser_outputs/m...
7,slam_llm.pdf,mineru,success,88200,31.78,structure_aware,mineru CLI,,/home/jovyan/outputs/slam_llm/parser_outputs/m...
8,slam_llm.pdf,grobid,success,69846,1.97,structured_metadata,GROBID server at localhost:8070,,/home/jovyan/outputs/slam_llm/parser_outputs/g...
9,cse188_research_paper.pdf,pymupdf,success,31929,0.01,flat_text,,,/home/jovyan/outputs/cse188_research_paper/par...
